# 02. Data Preprocessing & Feature Engineering

This notebook handles dataset deduplication, domain feature engineering, binary target formulation, stratified train-test splitting, and feature scaling.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


## 1. Load Data and Deduplication


In [ ]:
data_path = Path("../data/raw/winequality-red.csv") if Path("../data/raw/winequality-red.csv").exists() else Path("data/raw/winequality-red.csv")
df = pd.read_csv(data_path, sep=";")
df.columns = [col.strip().strip('"') for col in df.columns]

print(f"Raw records: {len(df)}")
df_cleaned = df.drop_duplicates().reset_index(drop=True)
print(f"Cleaned records: {len(df_cleaned)} (Removed {len(df) - len(df_cleaned)} duplicates)")


## 2. Domain Feature Engineering


In [ ]:
def engineer_features(df_in):
    df_out = df_in.copy()
    # Total acidity combination
    df_out["total_acidity"] = df_out["fixed acidity"] + df_out["volatile acidity"] + df_out["citric acid"]
    # Bound SO2
    df_out["bound_sulfur_dioxide"] = np.maximum(0.0, df_out["total sulfur dioxide"] - df_out["free sulfur dioxide"])
    # Ratio of free to total sulfur dioxide
    free_ratio = df_out["free sulfur dioxide"] / (df_out["total sulfur dioxide"] + 1e-5)
    df_out["free_sulfur_ratio"] = np.clip(free_ratio, 0.0, 1.0)
    # Alcohol to residual sugar ratio
    df_out["alcohol_to_sugar_ratio"] = np.maximum(0.0, df_out["alcohol"] / (df_out["residual sugar"] + 1e-5))
    return df_out

df_features = engineer_features(df_cleaned)
df_features.head()


## 3. Formulate Target Variable (Binary Classification)


In [ ]:
# High quality (score >= 7) vs Standard/Poor (score < 7)
X = df_features.drop(columns=["quality"])
y = (df_features["quality"] >= 7).astype(int)

print("Target Class Distribution:")
print(y.value_counts(normalize=True))


## 4. Stratified Train-Test Split and Scaling


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"X_train shape: {X_train_scaled.shape}, X_test shape: {X_test_scaled.shape}")


## 5. Export Processed Datasets


In [ ]:
processed_dir = Path("../data/processed") if Path("../data/processed").parent.exists() else Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

df_cleaned.to_csv(processed_dir / "winequality-cleaned.csv", index=False)

train_export = X_train_scaled.copy()
train_export["quality_target"] = y_train.values
train_export.to_csv(processed_dir / "train_scaled.csv", index=False)

test_export = X_test_scaled.copy()
test_export["quality_target"] = y_test.values
test_export.to_csv(processed_dir / "test_scaled.csv", index=False)

print("Successfully saved processed datasets to data/processed/")
